# Data Preparation, Deterministic Splits, and GBDT Tuning

This notebook documents the dataset acquisition route, the exact
preparation operations used for this release, the train/test and
out-of-fold assignments, and the fixed GBDT tuning procedure.

Run the cells from top to bottom in Google Colab. The released
prepared dataset is used by default. Kaggle download is optional
because it requires the user's own Kaggle credentials.


## 1. Connect Google Drive and install the environment


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os

configured_root = os.environ.get("LICE_PROJECT_ROOT")
project_candidates = [
    Path(configured_root) if configured_root else None,
    Path.cwd(),
    Path("/content/drive/MyDrive/Research/LICE_Guided_Model_Refinement_v1.3.0"),
    Path("/content/drive/MyDrive/LICE_Guided_Model_Refinement_v1.3.0"),
]
PROJECT_ROOT = next((p.resolve() for p in project_candidates if p and (p / "data").exists()), None)
assert PROJECT_ROOT is not None, "Repository not found; set LICE_PROJECT_ROOT."

from importlib.metadata import PackageNotFoundError, version
import sys

PINNED_BINARY_STACK = {
    "numpy": ("numpy", "2.2.6"),
    "pandas": ("pandas", "2.3.3"),
    "scikit-learn": ("sklearn", "1.5.2"),
    "scipy": ("scipy", "1.15.3"),
    "statsmodels": ("statsmodels", "0.14.4"),
}

def installed_version(distribution_name):
    try:
        return version(distribution_name)
    except PackageNotFoundError:
        return None

restart_required = any(
    installed_version(distribution) != required
    or (
        module in sys.modules
        and getattr(sys.modules[module], "__version__", None) != required
    )
    for distribution, (module, required) in PINNED_BINARY_STACK.items()
)

%pip install -q -r "{PROJECT_ROOT / 'environment' / 'requirements_oof.txt'}"

if restart_required:
    print(
        "Pinned packages were installed. Colab will restart now. "
        "After it reconnects, run this setup cell once more and "
        "then continue to the next cell.",
        flush=True,
    )
    os.kill(os.getpid(), 9)

print(f"Release folder: {PROJECT_ROOT}")


## 2. Imports, constants, and version record


In [ ]:
from importlib.metadata import version
from pathlib import Path
import hashlib
import json
import platform
import shutil
import subprocess

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)

try:
    from IPython.display import display
except ImportError:
    display = print

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "test_ablation"
SPLITS_DIR = PROJECT_ROOT / "splits"
for directory in (DATA_DIR, RESULTS_DIR, SPLITS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RAW_FILENAME = "diabetes_binary_5050split_health_indicators_BRFSS2015.csv"
KAGGLE_DATASET = "alexteboul/diabetes-health-indicators-dataset"
PREPARED_PATH = DATA_DIR / "diabetes_brfss2015_prepared.csv"
EXPECTED_PREPARED_SHA256 = (
    "cea4e25cd6304a5f35f5f71cd1b374b"
    "ef9613ecb9e1ff2a9b2056c7a3d8b7cc8"
)

RANDOM_SEED = 42
TEST_SIZE = 0.20
OOF_FOLDS = 3
DECISION_THRESHOLD = 0.50

def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(chunk_size), b""):
            digest.update(block)
    return digest.hexdigest()

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 17)

package_versions = {
    name: version(name)
    for name in [
        "numpy", "pandas", "scikit-learn", "scipy", "statsmodels"
    ]
}
print("Python:", platform.python_version())
display(pd.DataFrame(package_versions.items(), columns=["Package", "Version"]))


## 3. Dataset acquisition

The original source is the Kaggle file
`diabetes_binary_5050split_health_indicators_BRFSS2015.csv` from
`alexteboul/diabetes-health-indicators-dataset`. It contains 70,692
rows and 21 predictor variables plus the binary diabetes label.

Set `DOWNLOAD_FROM_KAGGLE = True` only when Kaggle credentials are
configured in the Colab session. Otherwise, the notebook uses the
prepared release file already present under `data/`.


In [ ]:
DOWNLOAD_FROM_KAGGLE = False
RAW_DATA_DIR = DATA_DIR / "raw"
RAW_DATA_DIR.mkdir(exist_ok=True)
RAW_PATH = RAW_DATA_DIR / RAW_FILENAME

if DOWNLOAD_FROM_KAGGLE:
    subprocess.run(
        [
            "kaggle", "datasets", "download",
            "-d", KAGGLE_DATASET,
            "-p", str(RAW_DATA_DIR),
            "--unzip",
        ],
        check=True,
    )

if RAW_PATH.exists():
    source_data = pd.read_csv(RAW_PATH)
    print(f"Loaded original Kaggle source: {RAW_PATH.name}")
else:
    source_data = pd.read_csv(PREPARED_PATH)
    print(
        "Original Kaggle source not present; using the released "
        "prepared dataset for verification."
    )

print("Loaded shape:", source_data.shape)


## 4. Exact preparation operations

The preparation sequence is:

1. rename `Diabetes_binary` to `Outcome` when the Kaggle source is used;
2. remove exact duplicate rows, retaining the first occurrence;
3. for `BMI`, `MentHlth`, and `PhysHlth`, identify values outside
$[Q_1-1.5\,IQR,\;Q_3+1.5\,IQR]$ using linear quantiles and replace
   those values with that feature's median; and
4. retain the published column order and original retained-row order.

No case-level health identifiers are present in this public dataset.


In [ ]:
EXPECTED_COLUMNS = [
    "Outcome", "HighBP", "HighChol", "CholCheck", "BMI", "Smoker",
    "Stroke", "HeartDiseaseorAttack", "PhysActivity", "Fruits",
    "Veggies", "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost",
    "GenHlth", "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age",
    "Education", "Income",
]

def prepare_brfss(frame):
    prepared = frame.copy()
    if "Diabetes_binary" in prepared.columns:
        prepared = prepared.rename(columns={"Diabetes_binary": "Outcome"})
    prepared = prepared.drop_duplicates(keep="first")
    for feature in ["BMI", "MentHlth", "PhysHlth"]:
        q1 = prepared[feature].quantile(0.25, interpolation="linear")
        q3 = prepared[feature].quantile(0.75, interpolation="linear")
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        median = prepared[feature].median()
        mask = (prepared[feature] < lower) | (prepared[feature] > upper)
        prepared.loc[mask, feature] = median
    return prepared[EXPECTED_COLUMNS].reset_index(drop=True)

if RAW_PATH.exists():
    prepared = prepare_brfss(source_data)
    prepared.to_csv(PREPARED_PATH, index=False)
else:
    prepared = source_data[EXPECTED_COLUMNS].copy()

assert prepared.shape == (69057, 22)
assert prepared.isna().sum().sum() == 0
assert list(prepared.columns) == EXPECTED_COLUMNS
assert file_sha256(PREPARED_PATH) == EXPECTED_PREPARED_SHA256

print("Prepared dataset verified.")
print("Shape:", prepared.shape)
print("SHA-256:", file_sha256(PREPARED_PATH))
display(prepared.head())


## 5. Train/test and out-of-fold assignments


In [ ]:
X = prepared.drop(columns="Outcome")
y = prepared["Outcome"].astype(int)
source_indices = np.arange(len(prepared), dtype=int)

train_idx, test_idx = train_test_split(
    source_indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=None,
)

split_table = pd.DataFrame({
    "Source_Row_Index": source_indices,
    "Outer_Split": "",
    "Train_Position": pd.Series([pd.NA] * len(prepared), dtype="Int64"),
    "Test_Position": pd.Series([pd.NA] * len(prepared), dtype="Int64"),
    "OOF_Validation_Fold": pd.Series([pd.NA] * len(prepared), dtype="Int64"),
})
split_table.loc[train_idx, "Outer_Split"] = "train"
split_table.loc[test_idx, "Outer_Split"] = "test"
split_table.loc[train_idx, "Train_Position"] = np.arange(len(train_idx))
split_table.loc[test_idx, "Test_Position"] = np.arange(len(test_idx))

cv = StratifiedKFold(n_splits=OOF_FOLDS, shuffle=False)
y_train = y.iloc[train_idx].reset_index(drop=True)
for fold, (_, validation_positions) in enumerate(
    cv.split(np.zeros(len(train_idx)), y_train), start=1
):
    validation_source_indices = train_idx[validation_positions]
    split_table.loc[
        validation_source_indices, "OOF_Validation_Fold"
    ] = fold

SPLIT_PATH = SPLITS_DIR / "split_assignments.csv.gz"
split_table.to_csv(SPLIT_PATH, index=False, compression="gzip")

assert len(train_idx) == 55245
assert len(test_idx) == 13812
assert split_table["OOF_Validation_Fold"].notna().sum() == 55245

display(split_table.groupby("Outer_Split").size().rename("Rows"))
display(
    split_table.dropna(subset=["OOF_Validation_Fold"])
    .groupby("OOF_Validation_Fold").size().rename("Validation_rows")
)


## 6. Fixed GBDT tuning space and selection rule

The search uses `RandomizedSearchCV` with 10 sampled configurations,
three-fold stratified cross-validation, F1 as the selection score,
and random seed 42. The highest mean cross-validation F1 is selected;
scikit-learn's rank order resolves any exact tie.


In [ ]:
TUNING_SPACE = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 0.2],
    "max_depth": [3, 5, 7],
}
RUN_TUNING = True

if RUN_TUNING:
    search = RandomizedSearchCV(
        estimator=GradientBoostingClassifier(random_state=RANDOM_SEED),
        param_distributions=TUNING_SPACE,
        n_iter=10,
        scoring="f1",
        n_jobs=-1,
        cv=StratifiedKFold(n_splits=3, shuffle=False),
        refit=True,
        random_state=RANDOM_SEED,
        return_train_score=True,
    )
    search.fit(X.iloc[train_idx], y.iloc[train_idx])
    tuning_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
    tuning_results.to_csv(
        RESULTS_DIR / "tuning_cv_results_full_precision.csv",
        index=False,
        float_format="%.17g",
    )
    selection = {
        "best_parameters": search.best_params_,
        "best_mean_cv_f1": float(search.best_score_),
        "selection_rule": "highest mean three-fold CV F1",
        "scoring": "f1",
        "n_iter": 10,
        "random_seed": RANDOM_SEED,
    }
    with (RESULTS_DIR / "tuning_selection.json").open("w") as stream:
        json.dump(selection, stream, indent=2)
else:
    selection = json.loads(
        (RESULTS_DIR / "tuning_selection.json").read_text()
    )

assert selection["best_parameters"] == {
    "n_estimators": 100,
    "max_depth": 5,
    "learning_rate": 0.1,
}
display(selection)


## 7. Completion record


In [ ]:
preparation_record = {
    "dataset_source": KAGGLE_DATASET,
    "source_filename": RAW_FILENAME,
    "prepared_file": str(PREPARED_PATH.relative_to(PROJECT_ROOT)),
    "prepared_sha256": file_sha256(PREPARED_PATH),
    "prepared_shape": list(prepared.shape),
    "random_seed": RANDOM_SEED,
    "test_size": TEST_SIZE,
    "outer_split_stratified": False,
    "oof_folds": OOF_FOLDS,
    "oof_shuffle": False,
    "decision_threshold": DECISION_THRESHOLD,
    "package_versions": package_versions,
}
with (RESULTS_DIR / "data_split_and_tuning_record.json").open("w") as stream:
    json.dump(preparation_record, stream, indent=2)

print("Data preparation, split assignment, and tuning checks completed.")
